#Setup et chargement du dataset

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pyarabic.araby as ar
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report
from scipy.stats import wilcoxon
from collections import Counter
import copy
import matplotlib.pyplot as plt

# Charger le dataset ArSarcasm-v2
df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

print("✅ Dataset ArSarcasm-v2 chargé !")
print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")

In [ ]:
#Nettoyage du texte.

In [ ]:
def clean_tweet(tweet):
    tweet = re.sub(r'http\S+|www.\S+', '', tweet)
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    tweet = re.sub(r'[^ء-ي\s]', '', tweet)
    tweet = ar.normalize_hamza(tweet)
    tweet = ar.normalize_ligature(tweet)
    tweet = ar.strip_tashkeel(tweet)
    return tweet.strip()

df_train["cleaned_tweet"] = df_train["tweet"].apply(clean_tweet)
df_test["cleaned_tweet"] = df_test["tweet"].apply(clean_tweet)

print("✅ Nettoyage terminé !")
print(df_train[["tweet", "cleaned_tweet"]].head())

In [ ]:
#Extraction embeddings MARBERT en séquence complète.

In [ ]:
# ── Charger MARBERT ──
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERT")
model = AutoModel.from_pretrained("UBC-NLP/MARBERT")
model.eval()

print("✅ MARBERT chargé !")

# ── Vérification du choix MAX_LENGTH=64 ──
longueurs = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer.tokenize(t))
)
print(f"\nLongueur moyenne : {longueurs.mean():.1f} tokens")
print(f"95e percentile   : {longueurs.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs.quantile(0.99):.1f} tokens")
print("→ MAX_LENGTH=64 couvre >99% des tweets sans troncature")

In [ ]:
import os

fichiers_a_verifier = [
    "../outputs/train_embeddings_sequence.npy",
    "../outputs/test_embeddings_sequence.npy",
    "../outputs/train_labels.npy",
    "../outputs/test_labels.npy"
]

for fichier in fichiers_a_verifier:
    existe = os.path.exists(fichier)
    print(f"{'✅' if existe else '❌'} {fichier} → {'existe' if existe else 'NON trouvé'}")
    

In [ ]:
X_train_seq = np.load("../outputs/train_embeddings_sequence.npy")
X_test_seq = np.load("../outputs/test_embeddings_sequence.npy")

print("✅ Séquences déjà extraites, rechargées depuis outputs/ !")
print(f"Train : {X_train_seq.shape}")
print(f"Test  : {X_test_seq.shape}")


In [ ]:
#Charger les features pragmatiques

In [ ]:
train_prag_v2 = pd.read_csv("../outputs/train_pragmatic_v2.csv")
test_prag_v2 = pd.read_csv("../outputs/test_pragmatic_v2.csv")

feature_names_v2 = [
    'p_pos', 'p_neg', 'p_neu',
    'intensifier', 'irony', 'neg_pos', 'neg_neg',
    'hyperbole', 'exclamation', 'question', 'latin_ratio',
    'discriminant_vocab', 'hashtag_count', 'emoji_count',
    'interjection', 'quotation', 'repeated_chars', 'tweet_length'
]

X_train_prag = train_prag_v2.drop("label", axis=1).values
X_test_prag = test_prag_v2.drop("label", axis=1).values

y_train_full = np.load("../outputs/train_labels.npy")
y_test = np.load("../outputs/test_labels.npy")

print("✅ Features pragmatiques (18) chargées !")
print(f"X_train_prag : {X_train_prag.shape}")
print(f"X_test_prag  : {X_test_prag.shape}")
print(f"\nColonnes : {feature_names_v2}")

In [ ]:
#Split train / validation / test.

In [ ]:
indices = np.arange(len(y_train_full))

idx_train, idx_val = train_test_split(
    indices,
    test_size=0.15,
    stratify=y_train_full,
    random_state=42
)

X_train_final_seq = X_train_seq[idx_train]
X_val_seq = X_train_seq[idx_val]

X_train_final_prag = X_train_prag[idx_train]
X_val_prag = X_train_prag[idx_val]

y_train_final = y_train_full[idx_train]
y_val = y_train_full[idx_val]

print("✅ Split train/val/test créé !")
print(f"Train final : {X_train_final_seq.shape[0]} tweets")
print(f"Validation  : {X_val_seq.shape[0]} tweets")
print(f"Test        : {X_test_seq.shape[0]} tweets")

print(f"\nDistribution train_final : {np.bincount(y_train_final)}")
print(f"Distribution validation  : {np.bincount(y_val)}")
print(f"Distribution test        : {np.bincount(y_test)}")

In [ ]:
#Architecture SAtt-BLSTM-ConvNet. 

In [ ]:
class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128,
                 prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        
        # ── ConvNet 1D : extrait des patterns locaux (n-grammes) ──
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        
        # ── BiLSTM : lit la séquence dans les deux sens ──
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden,
                               batch_first=True, bidirectional=True)
        
        # ── Soft Attention : pondère les positions importantes ──
        self.attention = SoftAttention(lstm_hidden * 2)
        
        # ── Tête de classification (context + features pragmatiques) ──
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        
        out = self.relu(self.fc1(combined))
        out = self.fc2(out)
        return out

print("✅ Architecture SAtt-BLSTM-ConvNet définie !")


In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

def evaluer_test_complet(model, X_test_seq, X_test_prag, y_test):
    model.eval()
    with torch.no_grad():
        x_test_seq_t = torch.FloatTensor(X_test_seq)
        x_test_prag_t = torch.FloatTensor(X_test_prag)
        outputs = model(x_test_seq_t, x_test_prag_t)
        _, y_pred = torch.max(outputs, 1)
        y_pred = y_pred.numpy()
    
    # Métriques macro (moyenne équilibrée des 2 classes)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    precision_macro = precision_score(y_test, y_pred, average='macro')
    recall_macro = recall_score(y_test, y_pred, average='macro')
    accuracy = accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # Métriques spécifiques à la classe sarcastique (classe 1)
    f1_classe1 = f1_score(y_test, y_pred, pos_label=1)
    precision_classe1 = precision_score(y_test, y_pred, pos_label=1)
    recall_classe1 = recall_score(y_test, y_pred, pos_label=1)
    
    print(classification_report(y_test, y_pred))
    
    resultats = {
        "F1_macro": f1_macro,
        "Precision_macro": precision_macro,
        "Recall_macro": recall_macro,
        "Accuracy": accuracy,
        "MCC": mcc,
        "F1_sarcastique": f1_classe1,
        "Precision_sarcastique": precision_classe1,
        "Recall_sarcastique": recall_classe1
    }
    
    print(f"\n{'='*50}")
    print("RÉSUMÉ DES MÉTRIQUES")
    print(f"{'='*50}")
    for nom, valeur in resultats.items():
        print(f"{nom:25} : {valeur:.4f}")
    
    return resultats

print("✅ Fonction d'évaluation complète prête !")

In [ ]:
class SarcasmDataset(Dataset):
    def __init__(self, X_seq, X_prag, y):
        self.X_seq = torch.FloatTensor(X_seq)
        self.X_prag = torch.FloatTensor(X_prag)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_prag[idx], self.y[idx]


def entrainer_avec_early_stopping(X_seq, X_prag, y, X_val_seq, X_val_prag, y_val,
                                    lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDataset(X_seq, X_prag, y)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    class_counts = np.bincount(y)
    weights = torch.FloatTensor(1.0 / class_counts)
    
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            x_val_seq_t = torch.FloatTensor(X_val_seq)
            x_val_prag_t = torch.FloatTensor(X_val_prag)
            outputs = model(x_val_seq_t, x_val_prag_t)
            _, y_pred = torch.max(outputs, 1)
            y_pred = y_pred.numpy()
        f1_val = f1_score(y_val, y_pred, average='macro')
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{max_epochs} - Loss: {total_loss/len(train_loader):.4f} - F1 val: {f1_val:.4f}")
        
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        
        if epochs_sans_amelioration >= patience:
            print(f"   ⏹️ Early stopping à l'époque {epoch+1} (pas d'amélioration depuis {patience} époques)")
            break
    
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model


def evaluer_test(model, X_test_seq, X_test_prag, y_test):
    model.eval()
    with torch.no_grad():
        x_test_seq_t = torch.FloatTensor(X_test_seq)
        x_test_prag_t = torch.FloatTensor(X_test_prag)
        outputs = model(x_test_seq_t, x_test_prag_t)
        _, y_pred = torch.max(outputs, 1)
        y_pred = y_pred.numpy()
    
    f1 = f1_score(y_test, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test, y_pred)
    print(classification_report(y_test, y_pred))
    return f1, mcc


def evaluer_fige(model, X_seq, X_prag, y):
    model.eval()
    with torch.no_grad():
        x_seq_t = torch.FloatTensor(X_seq)
        x_prag_t = torch.FloatTensor(X_prag)
        outputs = model(x_seq_t, x_prag_t)
        _, y_pred = torch.max(outputs, 1)
        y_pred = y_pred.numpy()
    return f1_score(y, y_pred, average='macro')

print("✅ Fonctions d'entraînement et d'évaluation prêtes !")

In [ ]:
#Sélection des hyperparamètres

In [ ]:
def evaluer_hyperparam(lr, epochs=5, batch_size=32):
    torch.manual_seed(42)
    
    train_dataset = SarcasmDataset(X_train_final_seq, X_train_final_prag, y_train_final)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    class_counts = np.bincount(y_train_final)
    weights = torch.FloatTensor(1.0 / class_counts)
    
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=18)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x_seq, x_prag, y in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq, x_prag)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        x_val_seq_t = torch.FloatTensor(X_val_seq)
        x_val_prag_t = torch.FloatTensor(X_val_prag)
        outputs = model(x_val_seq_t, x_val_prag_t)
        _, y_pred = torch.max(outputs, 1)
        y_pred = y_pred.numpy()
    
    f1_val = f1_score(y_val, y_pred, average='macro')
    return f1_val, total_loss / len(train_loader)


print("⏳ Test de plusieurs learning rates sur le validation set...\n")
learning_rates = [0.001, 0.0001, 0.00001]
resultats_hyperparam = []

for lr in learning_rates:
    print(f"⏳ Test avec lr={lr}...")
    f1_val, loss_final = evaluer_hyperparam(lr, epochs=5)
    print(f"   F1 macro (validation) = {f1_val:.4f} | Loss finale = {loss_final:.4f}\n")
    resultats_hyperparam.append({"learning_rate": lr, "F1_validation": f1_val, "loss": loss_final})

hyperparam_df = pd.DataFrame(resultats_hyperparam)
hyperparam_df = hyperparam_df.sort_values("F1_validation", ascending=False)

print("="*50)
print("RÉSULTATS — Comparaison des learning rates")
print("="*50)
print(hyperparam_df.to_string(index=False))

meilleur_lr = hyperparam_df.iloc[0]["learning_rate"]
print(f"\n✅ Meilleur learning rate retenu : {meilleur_lr}")

In [ ]:
#5 runs SANS features pragmatiques (avec métriques complètes)

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans = []

X_train_prag_zeros = np.zeros_like(X_train_final_prag)
X_val_prag_zeros = np.zeros_like(X_val_prag)
X_test_prag_zeros = np.zeros_like(X_test_prag)

print("="*60)
print("5 RUNS — SANS features pragmatiques")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping(
        X_train_final_seq, X_train_prag_zeros, y_train_final,
        X_val_seq, X_val_prag_zeros, y_val,
        lr=0.001, max_epochs=50, patience=10
    )
    
    resultats_metriques = evaluer_test_complet(model_s, X_test_seq, X_test_prag_zeros, y_test)
    resultats_metriques["seed"] = seed
    resultats_5runs_sans.append(resultats_metriques)

print("\n✅ 5 runs SANS features terminés !")
pd.DataFrame(resultats_5runs_sans).to_csv("../outputs/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
#5 runs AVEC features pragmatiques 

In [ ]:
resultats_5runs_avec = []

print("="*60)
print("5 RUNS — AVEC features pragmatiques")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_a = entrainer_avec_early_stopping(
        X_train_final_seq, X_train_final_prag, y_train_final,
        X_val_seq, X_val_prag, y_val,
        lr=0.001, max_epochs=50, patience=10
    )
    
    resultats_metriques = evaluer_test_complet(model_a, X_test_seq, X_test_prag, y_test)
    resultats_metriques["seed"] = seed
    resultats_5runs_avec.append(resultats_metriques)

print("\n✅ 5 runs AVEC features terminés !")
pd.DataFrame(resultats_5runs_avec).to_csv("../outputs/5runs_avec_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
#le test de Wilcoxon sur F1_macro

In [ ]:
f1_sans_5runs = [r["F1_macro"] for r in resultats_5runs_sans]
f1_avec_5runs = [r["F1_macro"] for r in resultats_5runs_avec]

stat, p_value = wilcoxon(f1_avec_5runs, f1_sans_5runs, alternative='greater')

print(f"F1 SANS : {np.mean(f1_sans_5runs):.4f} ± {np.std(f1_sans_5runs):.4f}")
print(f"F1 AVEC : {np.mean(f1_avec_5runs):.4f} ± {np.std(f1_avec_5runs):.4f}")
print(f"\nWilcoxon unilatéral (H1: AVEC > SANS)")
print(f"Statistique : {stat:.4f} | P-value : {p_value:.4f}")

if p_value < 0.05:
    print("✅ SIGNIFICATIF")
elif p_value < 0.10:
    print("⚠️ MARGINALEMENT significatif")
else:
    print("⚠️ NON significatif")
    

In [ ]:
#Modèle de référence pour l'ablation

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_ablation = entrainer_avec_early_stopping(
    X_train_final_seq, X_train_final_prag, y_train_final,
    X_val_seq, X_val_prag, y_val,
    lr=0.001, max_epochs=50, patience=10
)

f1_base_ablation = evaluer_fige(model_ablation, X_test_seq, X_test_prag, y_test)
print(f"\n✅ F1 de référence pour l'ablation : {f1_base_ablation:.4f}")

In [ ]:
#LOFO sur les 18 features

In [ ]:
feature_names_v2 = [
    'p_pos', 'p_neg', 'p_neu',
    'intensifier', 'irony', 'neg_pos', 'neg_neg',
    'hyperbole', 'exclamation', 'question', 'latin_ratio',
    'discriminant_vocab', 'hashtag_count', 'emoji_count',
    'interjection', 'quotation', 'repeated_chars', 'tweet_length'
]

print(f"F1 de référence : {f1_base_ablation:.4f}\n")
print("⏳ LOFO en cours (rapide, pas de ré-entraînement)...")

resultats_lofo_v2 = []

for i, feature_name in enumerate(feature_names_v2):
    X_test_prag_zero = X_test_prag.copy()
    X_test_prag_zero[:, i] = 0
    
    f1 = evaluer_fige(model_ablation, X_test_seq, X_test_prag_zero, y_test)
    impact = f1_base_ablation - f1
    
    print(f"Sans {feature_name:20} → F1 = {f1:.4f} | Impact = {impact:+.4f}")
    resultats_lofo_v2.append({
        "Feature": feature_name,
        "F1 sans feature": round(f1, 4),
        "Impact": round(impact, 4)
    })

lofo_v2_df = pd.DataFrame(resultats_lofo_v2)
lofo_v2_df = lofo_v2_df.sort_values("Impact", ascending=False)

print("\n✅ LOFO terminé !")
print(lofo_v2_df.to_string())
lofo_v2_df.to_csv("../outputs/lofo_18features_final.csv", index=False)

In [ ]:
#le Group-Level Ablation (3 groupes)

In [ ]:
idx_sentiment = [0, 1, 2]  # p_pos, p_neg, p_neu
idx_lexical = [3, 4, 5, 6, 7, 8, 9, 10]  # 8 features originales
idx_nouvelles = [11, 12, 13, 14, 15, 16, 17]  # 7 nouvelles features

print(f"F1 de référence : {f1_base_ablation:.4f}\n")

resultats_group_v2 = []

for nom_groupe, indices in [
    ("Sentiment Proxy", idx_sentiment),
    ("Lexical-Pragmatique original", idx_lexical),
    ("Nouvelles features enrichies", idx_nouvelles)
]:
    X_test_prag_zero = X_test_prag.copy()
    X_test_prag_zero[:, indices] = 0
    
    f1 = evaluer_fige(model_ablation, X_test_seq, X_test_prag_zero, y_test)
    impact = f1_base_ablation - f1
    print(f"Sans groupe '{nom_groupe}' ({len(indices)} features) :")
    print(f"   F1 = {f1:.4f} | Impact = {impact:+.4f}\n")
    
    resultats_group_v2.append({
        "Groupe": nom_groupe,
        "Nb_features": len(indices),
        "F1_sans_groupe": round(f1, 4),
        "Impact": round(impact, 4)
    })

group_v2_df = pd.DataFrame(resultats_group_v2)
group_v2_df.to_csv("../outputs/group_ablation_18features_final.csv", index=False)
print("✅ Group-Level Ablation terminée et sauvegardée !")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import gc

gc.collect()

X_train_seq = np.load("../outputs/train_embeddings_sequence.npy", mmap_mode='r')
X_test_seq = np.load("../outputs/test_embeddings_sequence.npy", mmap_mode='r')

train_prag_v2_full = pd.read_csv("../outputs/train_pragmatic_v2.csv")
test_prag_v2_full = pd.read_csv("../outputs/test_pragmatic_v2.csv")

X_train_prag = train_prag_v2_full.drop("label", axis=1).values
X_test_prag = test_prag_v2_full.drop("label", axis=1).values

y_train_full = np.load("../outputs/train_labels.npy")
y_test = np.load("../outputs/test_labels.npy")

indices = np.arange(len(y_train_full))
idx_train, idx_val = train_test_split(
    indices, test_size=0.15, stratify=y_train_full, random_state=42
)
idx_test_all = np.arange(len(y_test))

print("✅ Données ArSarcasm-v2 rechargées !")

In [ ]:
class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128, prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = SoftAttention(lstm_hidden * 2)
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class SarcasmDatasetMemmap(Dataset):
    def __init__(self, X_seq_memmap, X_prag, y, indices):
        self.X_seq_memmap = X_seq_memmap
        self.X_prag = torch.FloatTensor(X_prag[indices])
        self.y = torch.LongTensor(y[indices])
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        seq = np.array(self.X_seq_memmap[self.indices[idx]])
        return torch.FloatTensor(seq), self.X_prag[idx], self.y[idx]

def entrainer_avec_early_stopping_memmap(X_seq_memmap, X_prag, y, idx_train, idx_val,
                                           lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y[idx_train])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    for epoch in range(max_epochs):
        model.train()
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_seq_b, x_prag_b, y_b in val_loader:
                outputs = model(x_seq_b, x_prag_b)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.numpy())
                all_labels.extend(y_b.numpy())
        f1_val = f1_score(all_labels, all_preds, average='macro')
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            break
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model

def evaluer_fige_features_modifiees(model, X_seq_memmap, X_prag_modif, y, idx):
    dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag_modif, y, idx)
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    return f1_score(all_labels, all_preds, average='macro')

print("✅ Architecture et fonctions redéfinies !")

In [ ]:
lofo_v2_df = pd.read_csv("../outputs/lofo_18features_final.csv")
print("✅ LOFO ArSarcasm-v2 rechargé !")
print(lofo_v2_df.to_string(index=False))

torch.manual_seed(42)
np.random.seed(42)

model_ablation = entrainer_avec_early_stopping_memmap(
    X_train_seq, X_train_prag, y_train_full,
    idx_train, idx_val,
    lr=0.001, max_epochs=50, patience=10
)

f1_base_ablation = evaluer_fige_features_modifiees(
    model_ablation, X_test_seq, X_test_prag, y_test, idx_test_all
)
print(f"\n✅ F1 de référence pour l'ablation : {f1_base_ablation:.4f}")

In [ ]:
import random
import matplotlib.pyplot as plt

feature_names_v2 = [
    'p_pos', 'p_neg', 'p_neu',
    'intensifier', 'irony', 'neg_pos', 'neg_neg',
    'hyperbole', 'exclamation', 'question', 'latin_ratio',
    'discriminant_vocab', 'hashtag_count', 'emoji_count',
    'interjection', 'quotation', 'repeated_chars', 'tweet_length'
]

ordre_lofo_v2 = lofo_v2_df["Feature"].tolist()
indices_ordre_v2 = [feature_names_v2.index(f) for f in ordre_lofo_v2]

def evaluer_cumulative_v2(ordre_indices):
    f1_scores = [f1_base_ablation]
    X_test_prag_modif = X_test_prag.copy()
    for idx in ordre_indices:
        X_test_prag_modif = X_test_prag_modif.copy()
        X_test_prag_modif[:, idx] = 0
        f1 = evaluer_fige_features_modifiees(
            model_ablation, X_test_seq, X_test_prag_modif, y_test, idx_test_all
        )
        f1_scores.append(f1)
    return f1_scores

print("⏳ Courbe dans l'ordre d'importance...")
f1_courbe_importance_v2 = evaluer_cumulative_v2(indices_ordre_v2)
print("✅ Terminé !")

random.seed(42)
courbes_aleatoires_v2 = []
for run in range(5):
    ordre_alea = list(range(18))
    random.shuffle(ordre_alea)
    courbe = evaluer_cumulative_v2(ordre_alea)
    courbes_aleatoires_v2.append(courbe)
    print(f"Run aléatoire {run+1}/5 terminé")

labels_x = ["Référence"] + ordre_lofo_v2

courbes_array_v2 = np.array(courbes_aleatoires_v2)
moyenne_alea_v2 = courbes_array_v2.mean(axis=0)
std_alea_v2 = courbes_array_v2.std(axis=0)
etapes = list(range(19))

fig, ax = plt.subplots(figsize=(14, 7))
ax.fill_between(etapes, moyenne_alea_v2 - std_alea_v2, moyenne_alea_v2 + std_alea_v2,
                color='grey', alpha=0.3, label='Ordre aléatoire (moyenne ± std)')
ax.plot(etapes, moyenne_alea_v2, color='grey', linestyle='--', linewidth=1)
ax.plot(etapes, f1_courbe_importance_v2, color='red', marker='o', linewidth=2,
        label="Ordre d'importance (LOFO)")
ax.axhline(y=f1_base_ablation, color='blue', linestyle=':', linewidth=1.5,
           label=f'F1 complet = {f1_base_ablation:.4f}')

ax.set_xlabel('Feature retirée (cumulativement)', labelpad=15)
ax.set_ylabel('F1 macro')
ax.set_title('Ablation cumulative — MARBERT, ArSarcasm-v2 (18 features)')
ax.legend()
ax.set_xticks(etapes)
ax.set_xticklabels(labels_x, rotation=90, ha='center', fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/cumulative_ablation_noms.png", dpi=150)
plt.show()
print("✅ Graphique avec noms sauvegardé !")